In [27]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, random_split
from torchtext.data.utils import get_tokenizer
from collections import defaultdict
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data  = dataset["test"]

print(Counter(train_data["label"]))  
print(Counter(test_data["label"])) 


Counter({0: 12500, 1: 12500})
Counter({0: 12500, 1: 12500})


In [28]:
tokenizer= get_tokenizer("basic_english")

vocab= defaultdict(lambda: 0)

counter=1

for text in dataset["train"]["text"]:
    for token in tokenizer(text):
        if token not in vocab:
            vocab[token]= counter
            counter+=1

vocab_size= len(vocab) + 1


def text_pipeline(text, vocab=vocab):
    return [vocab[token] for token in tokenizer(text)]
    

In [29]:
class SentimentClassifier(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,output_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embedding_dim)
        self.lstm= nn.LSTM(embedding_dim, hidden_dim, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc= nn.Linear(hidden_dim*2, output_dim)
    
    def forward(self, text):

        embedding= self.embedding(text)
        output, (h_n,c_n)= self.lstm(embedding)
        forward_hidden_state= h_n[0]
        backward_hidden_state= h_n[1]
        last_hidden_state = torch.cat([forward_hidden_state, backward_hidden_state], dim=1)
        out= self.fc(last_hidden_state)

        return out 

device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= SentimentClassifier(vocab_size=vocab_size,embedding_dim=100, hidden_dim=256, output_dim=1)
model= model.to(device)


/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [30]:
def collate(batch):

    text_list, label_list= [],[]
    max_len= 256

    for item in batch:

        label_list.append(item['label'])
        processed_text= torch.tensor(text_pipeline(item["text"],vocab)[:max_len], dtype=torch.int64)
        text_list.append(processed_text)

    text_list= pad_sequence(text_list, batch_first=True, padding_value=0)
    label_list= torch.tensor(label_list)

    return label_list, text_list

train_size= int(0.9 * len(train_data))
val_size= len(train_data)- train_size

train_set, val_set= random_split(train_data,[train_size,val_size])

data_loader= DataLoader(train_set, batch_size=64,shuffle=True, collate_fn=collate)
test_loader= DataLoader(test_data,batch_size=64,shuffle=False, collate_fn=collate)
val_loader= DataLoader(val_set,batch_size=64,shuffle=False, collate_fn=collate)

In [31]:
optimizer= torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion= nn.BCEWithLogitsLoss()
scheduler= torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=10, eta_min=1e-4)

num_epochs=10
counter=0
patience=3

best_loss= float("inf")

for epoch in range(num_epochs):

    model.train()
    running_loss=0

    for batch_idx, (label,text) in enumerate(data_loader):

        label= label.float().unsqueeze(1).to(device)
        text=  text.to(device)

        output= model(text)
        loss= criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

        if (batch_idx+1) % 100 ==0:
            print(f"epoch {epoch+1}/{num_epochs} | batch {batch_idx+1}/{len(data_loader)} | loss {loss.item()}")
   
    average_loss= running_loss/len(data_loader)
    print(f"average loss of epoch {epoch+1}/{num_epochs} | : {average_loss}")
    
    with torch.no_grad():
        model.eval()
        running_val_loss=0

        for label,text in val_loader:

            label= label.float().unsqueeze(1).to(device)
            text= text.to(device)

            output= model(text)
            loss= criterion(output,label)

            running_val_loss+= loss.item()

    average_val_loss= running_val_loss/ len(val_loader)
    print(f"average validation loss of epoch {epoch+1}/{num_epochs} | : {average_val_loss}")

    scheduler.step(average_val_loss)

    if average_val_loss < best_loss:
        best_loss= average_val_loss
        counter=0
        torch.save(model.state_dict(), "best_parameters.pth")

    else:
        counter+=1
        if counter>=patience:
              print(f"Early stopping at epoch {epoch+1}")
              break



epoch 1/10 | batch 100/352 | loss 0.6666641235351562
epoch 1/10 | batch 200/352 | loss 0.653433620929718
epoch 1/10 | batch 300/352 | loss 0.6519001722335815
average loss of epoch 1/10 | : 0.6642650502987884
average validation loss of epoch 1/10 | : 0.7208867028355599


/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:156: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


epoch 2/10 | batch 100/352 | loss 0.5705629587173462
epoch 2/10 | batch 200/352 | loss 0.6643239855766296
epoch 2/10 | batch 300/352 | loss 0.6706196665763855
average loss of epoch 2/10 | : 0.6272251030260866
average validation loss of epoch 2/10 | : 0.6333818197250366
epoch 3/10 | batch 100/352 | loss 0.4786216616630554
epoch 3/10 | batch 200/352 | loss 0.5824061036109924
epoch 3/10 | batch 300/352 | loss 0.5540437698364258
average loss of epoch 3/10 | : 0.5736687781119888
average validation loss of epoch 3/10 | : 0.6153695315122605
epoch 4/10 | batch 100/352 | loss 0.5028499364852905
epoch 4/10 | batch 200/352 | loss 0.6115128993988037
epoch 4/10 | batch 300/352 | loss 0.48647937178611755
average loss of epoch 4/10 | : 0.48308047343214805
average validation loss of epoch 4/10 | : 0.5484121903777123
epoch 5/10 | batch 100/352 | loss 0.4415055215358734
epoch 5/10 | batch 200/352 | loss 0.3401236832141876
epoch 5/10 | batch 300/352 | loss 0.3033069670200348
average loss of epoch 5/10 | 

In [32]:
model.load_state_dict(torch.load("best_parameters.pth", map_location=device))
def predict_sentiment(text):
    
    with torch.no_grad():
        model.eval()

        text_tensor= torch.tensor(text_pipeline(text, vocab), dtype=torch.int64).unsqueeze(0).to(device)
        prediction= model(text_tensor)
        sentiment= 'positive' if prediction>0.5 else 'negative'

        return sentiment, prediction
    

reviews= [
    'The movie was really great',
    'I really loved the movie',
    'It was really bad',
    'it was okay at its best',
    'Its not bad, but I wouldnt watch it again',
    'I expected it to be terrible… and somehow it wasnt',
    'The acting was great, the plot was painful.'
    
]

for review in reviews:
    sentiment,prediction= predict_sentiment(review)

    print(f"review: {review}")
    print(f"sentiment: {sentiment}")


review: The movie was really great
sentiment: positive
review: I really loved the movie
sentiment: positive
review: It was really bad
sentiment: negative
review: it was okay at its best
sentiment: negative
review: Its not bad, but I wouldnt watch it again
sentiment: negative
review: I expected it to be terrible… and somehow it wasnt
sentiment: negative
review: The acting was great, the plot was painful.
sentiment: negative


In [33]:
total=0
correct=0
TrueP = 0
TrueN = 0
FalseP = 0
FalseN = 0

model.load_state_dict(torch.load("best_parameters.pth", map_location=device))

with torch.no_grad():
   model.eval()

   for label,text in test_loader:

      label= label.to(device).float()
      text=  text.to(device)

      output= model(text)

      probs= torch.sigmoid(output)
      preds= (probs>=0.5).float()

      total += label.size(0)
      correct += (preds.squeeze()==label).sum().item()

      TrueP += ((preds == 1) & (label == 1)).sum().item()
      TrueN += ((preds == 0) & (label == 0)).sum().item()
      FalseP += ((preds == 1) & (label == 0)).sum().item()
      FalseN += ((preds == 0) & (label == 1)).sum().item()

   conf_matrix = torch.tensor([[TrueP, FalseN],
                               [FalseP, TrueN]])
   accuracy= 100 * correct/total

print(accuracy)
print(conf_matrix)

78.552
tensor([[593868, 205172],
        [138612, 661388]])
